# Data Cleaning

Tài liệu này trình bày quy trình Khám phá (Data Profiling) và Làm sạch dữ liệu (Data Cleaning) cơ bản bước đầu cho toàn bộ các tập dữ liệu được Ban Tổ Chức (BTC) cung cấp. 

Mục tiêu của notebook này là rà soát cấu trúc, chuẩn hóa kiểu dữ liệu, xử lý các giá trị trùng lặp hoặc khuyết thiếu, qua đó xây dựng một bộ dữ liệu nền tảng sạch sẽ và đáng tin cậy, sẵn sàng phục vụ cho các bước Khám phá chuyên sâu (EDA) và Mô hình hóa tiếp theo của đội.

### Preparing the library

In [1]:
import pandas as pd
import os

## Phase 1: Data Understanding & Profiling
Xem và đánh giá tổng quan dữ liệu, chưa thao tác chỉnh sửa trên dữ liệu

### 1.1 Load data

In [2]:
DATA_PATH = '../dataset/01_raw/'
print("Loading data...")

# Master
df_products = pd.read_csv(DATA_PATH + 'products.csv', low_memory=False)
df_customers   = pd.read_csv(DATA_PATH + 'customers.csv', low_memory=False)
df_promotions  = pd.read_csv(DATA_PATH + 'promotions.csv', low_memory=False)
df_geography   = pd.read_csv(DATA_PATH + 'geography.csv', low_memory=False)

# Transaction
df_orders  = pd.read_csv(DATA_PATH + 'orders.csv', low_memory=False)
df_order_items = pd.read_csv(DATA_PATH + 'order_items.csv', low_memory=False)
df_payments  = pd.read_csv(DATA_PATH + 'payments.csv', low_memory=False)
df_shipments = pd.read_csv(DATA_PATH + 'shipments.csv', low_memory=False)
df_returns  = pd.read_csv(DATA_PATH + 'returns.csv', low_memory=False)
df_reviews  = pd.read_csv(DATA_PATH + 'reviews.csv', low_memory=False)

# Analytical
df_sales = pd.read_csv(DATA_PATH + 'sales.csv', low_memory=False)
# df_sample_submission = pd.read_csv(DATA_PATH + 'sample_submission.csv', low_memory=False)

# Operational
df_inventory = pd.read_csv(DATA_PATH + 'inventory.csv', low_memory=False)
df_web_traffic = pd.read_csv(DATA_PATH + 'web_traffic.csv', low_memory=False)

print("Load data successfully")

Loading data...


Load data successfully


### 1.2 Describe Data

#### 1.2.1 Product categories

In [3]:
print("The overall structure of data:\n")
df_products.info()
print("\nThe random 5 lines of data:")
df_products.sample(5)

The overall structure of data:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2412 entries, 0 to 2411
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    2412 non-null   int64  
 1   product_name  2412 non-null   object 
 2   category      2412 non-null   object 
 3   segment       2412 non-null   object 
 4   size          2412 non-null   object 
 5   color         2412 non-null   object 
 6   price         2412 non-null   float64
 7   cogs          2412 non-null   float64
dtypes: float64(2), int64(1), object(5)
memory usage: 150.9+ KB

The random 5 lines of data:


,product_id,product_name,category,segment,size,color,price,cogs
39,575,SaigonFlex UC-40,Streetwear,Everyday,XL,purple,11157.957055,10600.059202
185,952,HanoiStreet UC-15,Streetwear,Everyday,S,orange,3321.446256,3155.373944
999,1141,MekongFit UM-18,Streetwear,Balanced,M,black,10507.848261,9982.455848
1422,2102,PhoenixWear MA-01,Casual,All-weather,L,orange,6299.370000,3167.323236
598,2150,PhoenixWear UE-03,Streetwear,Performance,L,red,5932.945301,4969.434984


In [4]:
duplicate_data = df_products[df_products.duplicated(keep=False)].sort_values(by=df_products.columns.tolist())
if len(duplicate_data) > 0:
    print(f"{len(duplicate_data)} duplicated rows:")
    display(duplicate_data.head())

duplicate_primary_key = df_products[df_products['product_id'].duplicated(keep=False)].sort_values(by='product_id')
if len(duplicate_primary_key) > 0:
    print(f"{len(duplicate_primary_key)} duplicated product_id rows:")
    display(duplicate_primary_key.head())

print("\nDescriptive statistics of data:")
display(df_products.describe())
display(df_products.describe(include='O'))


Descriptive statistics of data:


,product_id,price,cogs
count,2412.000000,2412.000000,2412.000000
mean,1206.500000,4928.216231,3868.346732
std,696.428747,4776.737669,3878.584151
min,1.000000,9.056594,5.183829
25%,603.750000,59.444924,35.066367
50%,1206.500000,4399.605000,3184.934093
75%,1809.250000,7720.513784,5864.916462
max,2412.000000,40950.000000,38902.500000


,product_name,category,segment,size,color
count,2412,2412,2412,2412,2412
unique,2172,4,8,4,10
top,VietMode RP-06,Streetwear,Activewear,S,orange
freq,3,1320,598,603,242


##### Phân tích tổng quan: 
- **Data structure:** 2412 rows and 8 columns.
- **Datatype:** 
    - `product_id` đang ở kiểu `int64`. Tuy nhiên, đây là biến định danh và không có ý nghĩa toán học nên cần chuyển sang kiểu `string` để thuận tiện cho việc phân tích.
    - `product_name` đang ở kiểu `object`. Tuy nhiên, để thuận tiện cho việc phân tích và tối ưu bộ nhớ, chúng tôi sẽ chuyển sang kiểu `string`.
    - `category`, `segment`, `size` và `color` đang ở kiểu `object`. Tuy nhiên, số các giá trị unique trong các biến này tương đối ít so với số dòng trong toàn bộ dữ liệu nên cần chuyển sang kiểu `category` để thuận tiện trong việc phân loại.
    - `price` và `cogs` đang ở kiểu `float64` -> phù hợp.
- **Missing value:** Không có Missing value trong toàn bộ dữ liệu.
- **Duplicate value:** Không tồn tại các dòng có giá trị trùng lặp hoàn toàn. Tuy nhiên dựa trên bảng thống kê mô tả của `product_name`, chỉ có **2172** giá tri unique -> tồn tại các mặt hàng có id khác nhau nhưng tên hoàn toàn giống nhau. Đây là vấn đề cần kiểm tra lại xem liệu có phải cùng 1 mặt hàng nhưng lại bị định danh bởi đồng thời nhiều `product_id`.
- **Descriptive statistics:** 
  - Biến `price` (giá bán) và `cogs` (giá vốn) đều có **mean > median** → phân phối lệch phải, tồn tại một số sản phẩm có giá bán cao.
  - Độ lệch chuẩn của cả hai biến đều **lớn** và xấp xỉ mean. Dữ liệu phân tán mạnh, cho thấy sản phẩm thuộc nhiều phân khúc giá khác nhau.
  - Khoảng giá trị rất rộng:
    - `price`: từ ~9 đến ~40,950  
    - `cogs`: từ ~5 đến ~38,902  
    → Có khả năng tồn tại **outliers** hoặc sự khác biệt lớn giữa các nhóm sản phẩm.
  → Dữ liệu hợp lý về mặt kinh doanh nhưng có thể cần xử lý thêm **outliers** và có thể cân nhắc **log transformation** để phân tích sâu hơn.

In [5]:
duplicate_primary_key = df_products[df_products['product_name'].duplicated(keep=False)].sort_values(by='product_name')
if len(duplicate_primary_key) > 0:
    print(f"{len(duplicate_primary_key)} duplicated product_name rows:")
    display(duplicate_primary_key.head(10))

426 duplicated product_name rows:


,product_id,product_name,category,segment,size,color,price,cogs
745,380,LotusWear UE-01,Streetwear,Performance,S,red,34.036218,21.343813
645,280,LotusWear UE-01,Streetwear,Performance,S,red,12596.850000,11967.007500
646,281,LotusWear UE-02,Streetwear,Performance,M,black,15746.850000,8750.524545
746,381,LotusWear UE-02,Streetwear,Performance,M,black,21.651582,12.139172
647,282,LotusWear UE-03,Streetwear,Performance,L,orange,14486.850000,12008.149965
747,382,LotusWear UE-03,Streetwear,Performance,L,orange,11147.850000,10315.105605
648,283,LotusWear UE-04,Streetwear,Performance,XL,blue,24.024859,15.410534
748,383,LotusWear UE-04,Streetwear,Performance,XL,blue,7286.850000,6664.553010
649,284,LotusWear UE-05,Streetwear,Performance,S,white,31.046489,18.375704
749,384,LotusWear UE-05,Streetwear,Performance,S,white,12596.850000,10771.566435


##### Phân tích dữ liệu trùng lặp trong `product_name`:
- Có 426 dòng có `product_name` trùng nhau. Điều này là hợp lý vì một sản phẩm có thể có nhiều biến thể (size, color).
- Tuy nhiên, quan sát chi tiết cho thấy tồn tại các trường hợp cùng `product_name`, `category`, `segment`, `size`, `color` nhưng có giá (`price`) và giá vốn (`cogs`) chênh lệch rất lớn giữa các dòng.
- Đây là dấu hiệu của vấn đề về chất lượng dữ liệu, vấn đề này có thể gây:
  - Sai lệch trong phân tích doanh thu và lợi nhuận
  - Double counting khi join với các bảng khác
  - Sai lệch trong mô hình dự đoán

→ Cần kiểm tra và xem xét sau khi phân tích các bảng có liên hệ với bảng này.

#### 1.2.2 Customers

In [6]:
print("The overall structure of data:\n")
df_customers.info()
print("\nThe random 5 lines of data:")
df_customers.sample(5)

The overall structure of data:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121930 entries, 0 to 121929
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   customer_id          121930 non-null  int64 
 1   zip                  121930 non-null  int64 
 2   city                 121930 non-null  object
 3   signup_date          121930 non-null  object
 4   gender               121930 non-null  object
 5   age_group            121930 non-null  object
 6   acquisition_channel  121930 non-null  object
dtypes: int64(2), object(5)
memory usage: 6.5+ MB

The random 5 lines of data:


,customer_id,zip,city,signup_date,gender,age_group,acquisition_channel
40214,51811,28438,Cam Pha,2021-10-10,Male,35-44,email_campaign
29813,38428,25557,Thai Nguyen,2015-12-15,Male,18-24,organic_search
21690,27991,44119,Phu Ly,2019-09-05,Male,25-34,social_media
38465,49573,24363,Uong Bi,2022-10-21,Female,18-24,paid_search
89299,115370,53036,Tuy Hoa,2020-08-19,Male,55+,social_media


In [7]:
duplicate_data = df_customers[df_customers.duplicated(keep=False)].sort_values(by=df_customers.columns.tolist())
if len(duplicate_data) > 0:
    print(f"{len(duplicate_data)} duplicated rows:")
    display(duplicate_data.head())

duplicate_primary_key = df_customers[df_customers['customer_id'].duplicated(keep=False)].sort_values(by='customer_id')
if len(duplicate_primary_key) > 0:
    print(f"{len(duplicate_primary_key)} duplicated customer_id rows:")
    display(duplicate_primary_key.head())

print("\nDescriptive statistics of data:")
display(df_customers.describe())
display(df_customers.describe(include='O'))


Descriptive statistics of data:


,customer_id,zip
count,121930.000000,121930.000000
mean,78736.898663,50990.165595
std,45492.202886,26871.914605
min,1.000000,1001.000000
25%,39343.500000,28689.250000
50%,78784.500000,49835.000000
75%,118156.750000,73488.000000
max,157563.000000,99950.000000


,city,signup_date,gender,age_group,acquisition_channel
count,121930,121930,121930,121930,121930
unique,42,3941,3,5,6
top,Cam Pha,2022-06-02,Female,25-34,organic_search
freq,4398,78,59640,36342,36450


##### Phân tích tổng quan: 
- **Data structure:** 121930 rows and 7 columns.
- **Datatype:** 
    - `customer_id` và `zip` đang ở kiểu `int64`. Tuy nhiên, đây là các biến định danh và không có ý nghĩa toán học nên cần chuyển sang kiểu `string` để thuận tiện cho việc phân tích.
    - `gender`, `age_group`, `city` và `acquisition_channel` đang ở kiểu `object`. Tuy nhiên, số các giá trị unique trong các biến này tương đối ít so với số dòng trong toàn bộ dữ liệu nên cần chuyển sang kiểu `category` để thuận tiện trong việc phân loại.
    - `signup_date` đang ở kiểu `object`. Tuy nhiên, đây là biến định danh ngày nên cần chuyển về kiểu `datetime`  để thuận tiện cho việc phân tích.
- **Missing value:** Không có Missing value trong toàn bộ dữ liệu.
- **Duplicate value:** Không có Duplicated value trong toàn bộ dữ liệu.

#### 1.2.3 Promotional programs

In [8]:
print("The overall structure of data:\n")
df_promotions.info()
print("\nThe random 5 lines of data:")
df_promotions.sample(5)

The overall structure of data:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   promo_id             50 non-null     object 
 1   promo_name           50 non-null     object 
 2   promo_type           50 non-null     object 
 3   discount_value       50 non-null     float64
 4   start_date           50 non-null     object 
 5   end_date             50 non-null     object 
 6   applicable_category  10 non-null     object 
 7   promo_channel        50 non-null     object 
 8   stackable_flag       50 non-null     int64  
 9   min_order_value      50 non-null     int64  
dtypes: float64(1), int64(2), object(7)
memory usage: 4.0+ KB

The random 5 lines of data:


,promo_id,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value
12,PROMO-0013,Fall Launch 2015,percentage,10.0,2015-08-30,2015-10-01,NaN,email,1,0
30,PROMO-0031,Spring Sale 2019,percentage,12.0,2019-03-18,2019-04-17,NaN,online,1,150000
42,PROMO-0043,Fall Launch 2021,percentage,10.0,2021-08-30,2021-10-02,NaN,email,0,0
1,PROMO-0002,Mid-Year Sale 2013,percentage,18.0,2013-06-23,2013-07-22,NaN,online,0,0
8,PROMO-0009,Fall Launch 2014,percentage,10.0,2014-08-30,2014-10-01,NaN,all_channels,0,100000


In [9]:
duplicate_data = df_promotions[df_promotions.duplicated(keep=False)].sort_values(by=df_promotions.columns.tolist())
if len(duplicate_data) > 0:
    print(f"{len(duplicate_data)} duplicated rows:")
    display(duplicate_data.head())

duplicate_primary_key = df_promotions[df_promotions['promo_id'].duplicated(keep=False)].sort_values(by='promo_id')
if len(duplicate_primary_key) > 0:
    print(f"{len(duplicate_primary_key)} duplicated promo_id rows:")
    display(duplicate_primary_key.head())

print("\nDescriptive statistics of data:")
display(df_promotions.describe())
display(df_promotions.describe(include='O'))


Descriptive statistics of data:


,discount_value,stackable_flag,min_order_value
count,50.000000,50.000000,50.000000
mean,18.500000,0.240000,46000.000000
std,11.241777,0.431419,66116.779802
min,10.000000,0.000000,0.000000
25%,12.000000,0.000000,0.000000
50%,16.500000,0.000000,0.000000
75%,20.000000,0.000000,100000.000000
max,50.000000,1.000000,200000.000000


,promo_id,promo_name,promo_type,start_date,end_date,applicable_category,promo_channel
count,50,50,50,50,50,10,50
unique,50,50,2,50,50,2,5
top,PROMO-0001,Spring Sale 2013,percentage,2013-03-18,2013-04-17,Streetwear,all_channels
freq,1,1,45,1,1,5,19


##### Phân tích tổng quan: 
- **Data structure:** 50 rows and 10 columns.
- **Datatype:** 
    - `promo_id` và `promo_name` đang ở kiểu `object`. Tuy nhiên, để thuận tiện cho việc phân tích và tối ưu bộ nhớ, chúng tôi sẽ chuyển sang kiểu `string`.
    - `promo_type`, `applicable_category` và `promo_channel` đang ở kiểu `object`. Tuy nhiên, số các giá trị unique trong các biến này tương đối ít so với số dòng trong toàn bộ dữ liệu nên cần chuyển sang kiểu `category` để thuận tiện trong việc phân loại.
    - `start_date` và `end_date` đang ở kiểu `object`. Tuy nhiên, đây là biến định danh ngày nên cần chuyển về kiểu `datetime`  để thuận tiện cho việc phân tích.
    - `discount_value` đang ở kiểu `float64` -> phù hợp nhưng cần tách biến này thành 2 biến độc lập để không lưu trữ 2 loại giá trị cùng lúc (giá trị % và giá trị tiền tệ).
    - `min_order_value` đang ở kiểu `int64` -> phù hợp.
    - `stackable_flag` đang ở kiểu `int64`. Tuy nhiên với min = 0 và max = 1, rất có thể đây cũng là biến phân loại -> cần kiểm tra lại để cân nhắc chuyển sang kiểu phù hợp.
- **Missing value:** Tồn tại các Missing value ở biến `applicable_category`. Theo mô tả trong đề bài, dữ liệu bị thiếu ở cột này đại diện cho việc áp dụng cho tất cả danh mục. Do đó cần điền giá trị **All** vào các Missing value, biểu thị chương trình áp dụng cho tất cả danh mục.
- **Duplicate value:** Không có Duplicated value trong toàn bộ dữ liệu.
- **Descriptive statistics:** 
    - `discount_value` đang có thể đại diện cho giá trị % hoặc giá trị tiền tệ nên chưa thể đánh giá dựa trên bảng thống kê mô tả được. 
    - `min_order_value` có **median = 0** và **Q1 = 0** cho thấy hơn 50% chương trình giảm giá không yêu cầu giá trị đơn hàng tối thiểu. Tại **Q3 ≈ 100 000**, điều này cho thấy chỉ khoảng 25% chương trình yêu cầu giá trị đơn hàng tối thiểu đáng kể. Tại **max = 200 000**, cho thấy có những chương trình đặt ngưỡng rất cao, có thể nhắm đến các đơn hàng giá trị lớn.
    - `min_order_value` có **mean (~46,000)** cao hơn nhiều so với **median (0)** → phân phối lệch phải rất mạnh, do một số giá trị lớn kéo trung bình lên. Đồng thời, độ lệch chuẩn của `min_order_value` (~66,117) rất lớn, cho thấy dữ liệu phân tán mạnh, tồn tại nhiều mức ngưỡng khác nhau.
→ Nhìn chung, `min_order_value` không đồng đều, phân hóa rõ rệt giữa các chương trình không yêu cầu giá trị tối thiểu (0) và các chương trình yêu cầu giá trị tối thiểu cao. Điều này có thể phản ánh các chiến lược khuyến mãi khác nhau của doanh nghiệp.

In [10]:
print(df_promotions['stackable_flag'].value_counts())

stackable_flag
0    38
1    12
Name: count, dtype: int64


Kết quả cho thấy `stackable_flag` chỉ tồn tại 2 giá trị 0 và 1 -> là một biến phân loại như mong đợi, cần chuyển sang kiểu `bool`

#### 1.2.4 Geography

In [11]:
print("The overall structure of data:\n")
df_geography.info()
print("\nThe random 5 lines of data:")
df_geography.sample(5)

The overall structure of data:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39948 entries, 0 to 39947
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   zip       39948 non-null  int64 
 1   city      39948 non-null  object
 2   region    39948 non-null  object
 3   district  39948 non-null  object
dtypes: int64(1), object(3)
memory usage: 1.2+ MB

The random 5 lines of data:


,zip,city,region,district
6078,45387,Uong Bi,East,District #16
20283,72459,Phan Rang-Thap Cham,Central,District #25
7459,48237,Phu Ly,East,District #18
31160,75966,Kon Tum,Central,District #24
30854,62067,Quy Nhon,Central,District #29


In [12]:
duplicate_data = df_geography[df_geography.duplicated(keep=False)].sort_values(by=df_geography.columns.tolist())
if len(duplicate_data) > 0:
    print(f"{len(duplicate_data)} duplicated rows:")
    display(duplicate_data.head())

duplicate_primary_key = df_geography[df_geography['zip'].duplicated(keep=False)].sort_values(by='zip')
if len(duplicate_primary_key) > 0:
    print(f"{len(duplicate_primary_key)} duplicated zip rows:")
    display(duplicate_primary_key.head())

print("\nDescriptive statistics of data:")
display(df_geography.describe())
display(df_geography.describe(include='O'))


Descriptive statistics of data:


,zip
count,39948.000000
mean,50895.084735
std,27042.257341
min,1.000000
25%,28279.500000
50%,49876.500000
75%,73526.250000
max,99950.000000


,city,region,district
count,39948,39948,39948
unique,42,3,39
top,Cam Pha,East,District #25
freq,1403,18929,1597


##### Phân tích tổng quan: 
- **Data structure:** 39948 rows and 4 columns.
- **Datatype:** 
    - `zip` đang ở kiểu `int64`. Tuy nhiên, đây là biến định danh và không có ý nghĩa toán học nên cần chuyển sang kiểu `string` để thuận tiện cho việc phân tích.
    - `city`, `region` và `district` đang ở kiểu `object`. Tuy nhiên, số các giá trị unique trong các biến này tương đối ít so với số dòng trong toàn bộ dữ liệu nên cần chuyển sang kiểu `category` để thuận tiện trong việc phân loại.
- **Missing value:** Không có Missing value trong toàn bộ dữ liệu.
- **Duplicate value:** Không có Duplicated value trong toàn bộ dữ liệu.

#### 1.2.5 Orders

In [13]:
print("The overall structure of data:\n")
df_orders.info()
print("\nThe random 5 lines of data:")
df_orders.sample(5)

The overall structure of data:



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   order_id        646945 non-null  int64 
 1   order_date      646945 non-null  object
 2   customer_id     646945 non-null  int64 
 3   zip             646945 non-null  int64 
 4   order_status    646945 non-null  object
 5   payment_method  646945 non-null  object
 6   device_type     646945 non-null  object
 7   order_source    646945 non-null  object
dtypes: int64(3), object(5)
memory usage: 39.5+ MB

The random 5 lines of data:


,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source
359142,463206,2017-02-15,94456,73165,delivered,apple_pay,desktop,organic_search
521081,671996,2019-06-18,152249,89169,delivered,credit_card,mobile,referral
586416,756189,2021-04-18,23850,44413,delivered,credit_card,tablet,paid_search
270520,348936,2015-12-30,128799,77498,delivered,credit_card,tablet,organic_search
359722,463995,2017-02-17,6273,17109,delivered,paypal,desktop,paid_search


In [14]:
duplicate_data = df_orders[df_orders.duplicated(keep=False)].sort_values(by=df_orders.columns.tolist())
if len(duplicate_data) > 0:
    print(f"{len(duplicate_data)} duplicated rows:")
    display(duplicate_data.head())

duplicate_primary_key = df_orders[df_orders['order_id'].duplicated(keep=False)].sort_values(by='order_id')
if len(duplicate_primary_key) > 0:
    print(f"{len(duplicate_primary_key)} duplicated order_id rows:")
    display(duplicate_primary_key.head())

print("\nDescriptive statistics of data:")
display(df_orders.describe())
display(df_orders.describe(include='O'))


Descriptive statistics of data:


,order_id,customer_id,zip
count,646945.000000,646945.000000,646945.000000
mean,417189.470332,84906.203535,55410.740423
std,240785.704463,48446.922752,28876.471824
min,1.000000,1.000000,1001.000000
25%,208728.000000,41336.000000,30904.000000
50%,417211.000000,87279.000000,54129.000000
75%,625628.000000,133282.000000,83301.000000
max,834397.000000,157563.000000,99950.000000


,order_date,order_status,payment_method,device_type,order_source
count,646945,646945,646945,646945,646945
unique,3833,6,5,3,6
top,2018-05-30,delivered,credit_card,mobile,organic_search
freq,803,516716,356352,291482,181495


##### Phân tích tổng quan: 
- **Data structure:** 646945 rows and 8 columns.
- **Datatype:** 
    - `order_id`, `customer_id`, và `zip` đang ở kiểu `int64`. Tuy nhiên, đây là các biến định danh và không có ý nghĩa toán học nên cần chuyển sang kiểu `string` để thuận tiện cho việc phân tích.
    - `order_status`, `payment_method`, `device_type` và `order_source` đang ở kiểu `object`. Tuy nhiên, số các giá trị unique trong các biến này tương đối ít so với số dòng trong toàn bộ dữ liệu nên cần chuyển sang kiểu `category` để thuận tiện trong việc phân loại.
    - `order_date` đang ở kiểu `object`. Tuy nhiên, đây là biến định danh ngày nên cần chuyển về kiểu `datetime`  để thuận tiện cho việc phân tích.
- **Missing value:** Không có Missing value trong toàn bộ dữ liệu.
- **Duplicate value:** Không có Duplicated value trong toàn bộ dữ liệu.

#### 1.2.6 Order detail items

In [15]:
print("The overall structure of data:\n")
df_order_items.info()
print("\nThe random 5 lines of data:")
df_order_items.sample(5)

The overall structure of data:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 714669 entries, 0 to 714668
Data columns (total 7 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   order_id         714669 non-null  int64  
 1   product_id       714669 non-null  int64  
 2   quantity         714669 non-null  int64  
 3   unit_price       714669 non-null  float64
 4   discount_amount  714669 non-null  float64
 5   promo_id         276316 non-null  object 
 6   promo_id_2       206 non-null     object 
dtypes: float64(2), int64(3), object(2)
memory usage: 38.2+ MB

The random 5 lines of data:


,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2
494736,569249,426,3,9852.82,3547.02,PROMO-0027,NaN
426423,489365,778,6,1496.25,0.00,NaN,NaN
129469,146184,2371,3,6953.59,0.00,NaN,NaN
527985,609332,965,3,5019.23,2710.38,PROMO-0028,NaN
309966,353288,2212,1,1098.38,0.00,NaN,NaN


In [16]:
duplicate_data = df_order_items[df_order_items.duplicated(keep=False)].sort_values(by=df_order_items.columns.tolist())
if len(duplicate_data) > 0:
    print(f"{len(duplicate_data)} duplicated rows:")
    display(duplicate_data.head())

print("\nDescriptive statistics of data:")
display(df_order_items.describe())
display(df_order_items.describe(include='O'))


Descriptive statistics of data:


,order_id,product_id,quantity,unit_price,discount_amount
count,714669.000000,714669.000000,714669.000000,714669.000000,714669.000000
mean,411615.076561,1234.931370,4.495988,5114.690157,1048.887415
std,240480.310686,691.332564,2.290143,3774.817912,2280.530606
min,1.000000,1.000000,1.000000,392.570000,0.000000
25%,203229.000000,689.000000,2.000000,1906.890000,0.000000
50%,409306.000000,990.000000,4.000000,4257.770000,0.000000
75%,618981.000000,2045.000000,6.000000,7273.760000,967.630000
max,834397.000000,2412.000000,8.000000,43056.000000,35235.470000


,promo_id,promo_id_2
count,276316,206
unique,50,2
top,PROMO-0014,PROMO-0015
freq,11451,132


##### Phân tích tổng quan: 
- **Data structure:** 714669 rows and 7 columns.
- **Datatype:** 
    - `order_id`, `product_id`,`promo_id` và `promo_id_2` đang ở kiểu `int64` và `object`. Tuy nhiên, đây là các biến định danh và không có ý nghĩa toán học nên cần chuyển sang kiểu `string` để tối ưu bộ nhớ và thuận tiện cho việc phân tích.
    - `unit_price` và `discount_amount` đang ở kiểu `float64` -> phù hợp.
    - `quantity` đang ở kiểu `int64` -> phù hợp.
- **Missing value:** Tồn tại các Missing value trong 2 cột `promo_id` và `promo_id_2`. Nguyên nhân là do các order_item này không áp dụng khuyến mãi. Do đó cần điền giá trị **None** vào các Missing value, biểu thị không có chương trình nào được áp dụng.
- **Duplicate value:** Không có Duplicated value trong toàn bộ dữ liệu.
- **Descriptive statistics:** 
  - `unit_price` có **mean > median** → phân phối lệch phải, tồn tại một số sản phẩm có giá bán cao. Độ lệch chuẩn của `unit_price` (**~3775**) khá lớn so với mean, cho thấy giá bán phân tán mạnh, phản ánh nhiều phân khúc sản phẩm.
  - Liên hệ với `price` ở bảng *products.csv*:
    - Min `unit_price` (~392)  > Min `price` (~9). Điều này có thấy có thể không phải tất cả các sản phẩm đều được bán mà chỉ có các sản phẩm từ phân khúc giá ~390 trở lên mới được bán.
    - Max `unit_price` (~43000)  > Max `price` (~41000). Điều này cho thấy rất có thể `unit_price` là giá bán động xung quanh `price`.

  - `discount_amount` có **median = 0** và **Q1 = 0** cho thấy hơn 50% đơn hàng không áp dụng chương trình giảm giá. Tại **Q3 ≈ 967**, điều này cho thấy chỉ khoảng 25% đơn hàng có giảm giá đáng kể.
  - `discount_amount` có **mean (~1049) > median (0)** → phân phối lệch phải rất mạnh. Đồng thời **max (~35235)** rất lớn, có thể tồn tại các chương trình giảm giá sâu hoặc outliers.
→ Dữ liệu hợp lý về mặt kinh doanh nhưng có thể cần xử lý thêm **outliers**  của `discount_amount` và có thể phân tích sâu để tìm hiểu liệu `unit_price` có phải là giá bán động xung quanh `price`.

#### 1.2.7 Payments

In [17]:
print("The overall structure of data:\n")
df_payments.info()
print("\nThe random 5 lines of data:")
df_payments.sample(5)

The overall structure of data:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 4 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   order_id        646945 non-null  int64  
 1   payment_method  646945 non-null  object 
 2   payment_value   646945 non-null  float64
 3   installments    646945 non-null  int64  
dtypes: float64(1), int64(2), object(1)
memory usage: 19.7+ MB

The random 5 lines of data:


,order_id,payment_method,payment_value,installments
40003,51688,paypal,9592.90,3
517770,667730,credit_card,18814.44,6
611367,788390,cod,23519.60,1
13443,17455,credit_card,39721.55,12
598294,771432,paypal,64773.40,1


In [18]:
duplicate_data = df_payments[df_payments.duplicated(keep=False)].sort_values(by=df_payments.columns.tolist())
if len(duplicate_data) > 0:
    print(f"{len(duplicate_data)} duplicated rows:")
    display(duplicate_data.head())

duplicate_primary_key = df_payments[df_payments['order_id'].duplicated(keep=False)].sort_values(by='order_id')
if len(duplicate_primary_key) > 0:
    print(f"{len(duplicate_primary_key)} duplicated order_id rows:")
    display(duplicate_primary_key.head())

print("\nDescriptive statistics of data:")
display(df_payments.describe())
display(df_payments.describe(include='O'))


Descriptive statistics of data:


,order_id,payment_value,installments
count,646945.000000,646945.000000,646945.000000
mean,417189.470332,24238.334426,3.448319
std,240785.704463,22378.475324,3.119582
min,1.000000,389.740000,1.000000
25%,208728.000000,7681.060000,1.000000
50%,417211.000000,17229.440000,3.000000
75%,625628.000000,33706.350000,6.000000
max,834397.000000,331570.400000,12.000000


,payment_method
count,646945
unique,5
top,credit_card
freq,356352


##### Phân tích tổng quan: 
- **Data structure:** 646945 rows and 4 columns.
- **Datatype:** 
    - `order_id` đang ở kiểu `int64`. Tuy nhiên, đây là các biến định danh và không có ý nghĩa toán học nên cần chuyển sang kiểu `string` để thuận tiện cho việc phân tích.
    - `payment_method` đang ở kiểu `object`. Tuy nhiên, số các giá trị unique trong các biến này tương đối ít so với số dòng trong toàn bộ dữ liệu nên cần chuyển sang kiểu `category` để thuận tiện trong việc phân loại.
    - `payment_value` đang ở kiểu `float64` -> phù hợp.
    - `installments` đang ở kiểu `int64`. Tuy nhiên với min = 1 và max = 12, rất có thể đây cũng là biến phân loại -> cần kiểm tra lại để cân nhắc chuyển sang kiểu `category`.
- **Missing value:** Không có Missing value trong toàn bộ dữ liệu.
- **Duplicate value:** Không có Duplicated value trong toàn bộ dữ liệu.
- **Descriptive statistics:** 
    - `payment_value` có **mean > median** (~24 238 > ~17 229) → phân phối lệch phải, tồn tại một số đơn hàng giá trị cao. Bên cạnh đó, độ lệch chuẩn lớn (~22 378) gần bằng mean cho thấy dữ liệu phân tán mạnh, có thể do sự đa dạng trong giá trị thanh toán giữa các đơn hàng.
    - `payment_value` có khoảng giá trị rộng với **min ≈ 390** và **max ≈ 331 570**. Rất có thể tồn tại các outliers vì phân vị cho thấy **50%** đơn hàng nằm trong khoảng **(~7 681 → ~33 706)**. Đồng thời, **median < mean** cho thấy phần lớn đơn hàng có giá trị vừa phải, chỉ một số ít đơn hàng lớn kéo trung bình lên.
→ Nhìn chung, `payment_value` có phân phối lệch phải và biến động lớn. Điều này cho thấy sự đa dạng về giá trị các đơn hàng và cần xử lý thêm **outliers**, cân nhắc **log transformation** để phân tích sâu hơn.

In [19]:
print(df_payments['installments'].value_counts())

installments
1     262866
3     218949
6     109910
12     54126
2       1094
Name: count, dtype: int64


Kết quả cho thấy `installments` chỉ tồn tại 5 giá trị distinct -> là một biến phân loại như mong đợi, cần chuyển sang kiểu `category`

#### 1.2.8 Shipments

In [20]:
print("The overall structure of data:\n")
df_shipments.info()
print("\nThe random 5 lines of data:")
df_shipments.sample(5)

The overall structure of data:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 566067 entries, 0 to 566066
Data columns (total 4 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   order_id       566067 non-null  int64  
 1   ship_date      566067 non-null  object 
 2   delivery_date  566067 non-null  object 
 3   shipping_fee   566067 non-null  float64
dtypes: float64(1), int64(1), object(2)
memory usage: 17.3+ MB

The random 5 lines of data:


,order_id,ship_date,delivery_date,shipping_fee
83593,122936,2013-10-19,2013-10-26,31.07
170277,250290,2015-02-11,2015-02-14,0.73
197670,290429,2015-06-04,2015-06-07,2.40
109091,160510,2014-04-02,2014-04-04,2.32
559475,823468,2022-09-06,2022-09-10,2.56


In [21]:
duplicate_data = df_shipments[df_shipments.duplicated(keep=False)].sort_values(by=df_shipments.columns.tolist())
if len(duplicate_data) > 0:
    print(f"{len(duplicate_data)} duplicated rows:")
    display(duplicate_data.head())

duplicate_primary_key = df_shipments[df_shipments['order_id'].duplicated(keep=False)].sort_values(by='order_id')
if len(duplicate_primary_key) > 0:
    print(f"{len(duplicate_primary_key)} duplicated order_id rows:")
    display(duplicate_primary_key.head())

print("\nDescriptive statistics of data:")
display(df_shipments.describe())
display(df_shipments.describe(include='O'))


Descriptive statistics of data:


,order_id,shipping_fee
count,566067.000000,566067.000000
mean,415816.869664,4.962857
std,240007.311562,8.887355
min,1.000000,0.000000
25%,208192.500000,0.870000
50%,415866.000000,1.730000
75%,623218.500000,2.600000
max,834325.000000,32.000000


,ship_date,delivery_date
count,566067,566067
unique,3831,3831
top,2018-06-02,2018-06-06
freq,678,562


##### Phân tích tổng quan: 
- **Data structure:** 5566067 rows and 4 columns.
- **Datatype:** 
    - `order_id` đang ở kiểu `int64`. Tuy nhiên, đây là các biến định danh và không có ý nghĩa toán học nên cần chuyển sang kiểu `string` để thuận tiện cho việc phân tích.
    - `ship_date` và `delivery_date` đang ở kiểu `object`. Tuy nhiên, đây là biến định danh ngày nên cần chuyển về kiểu `datetime`  để thuận tiện cho việc phân tích.
    - `shipping_fee` đang ở kiểu `float64` -> phù hợp.
- **Missing value:** Không có Missing value trong toàn bộ dữ liệu.
- **Duplicate value:** Không có Duplicated value trong toàn bộ dữ liệu.
- **Descriptive statistics:** 
    - `shipping_fee` có **mean > median** (~4.96 > ~1.73) → phân phối lệch phải, tồn tại một số đơn hàng có phí vận chuyển cao. Bên cạnh đó, độ lệch chuẩn (~8.89) lớn hơn cả mean → dữ liệu phân tán mạnh, có khả năng tồn tại outliers.
    - `shipping_fee` có khoảng giá trị **min ≈ 0** và **max ≈ 32**. Tồn tại các đơn hàng được miễn phí vận chuyển. 
    - Phân vị cho thấy cho thấy **50%** phí ship đơn hàng nằm trong khoảng **(~0.87 → ~2.60)**. Đồng thời, **median < mean** cho thấy phần lớn phần lớn đơn hàng có phí vận chuyển thấp, chỉ một số ít giá trị cao kéo trung bình lên.

→ Nhìn chung, `shipping_fee` có phân phối lệch phải do một số đơn hàng có phí cao, phí ship chủ yếu ở mức thấp hoặc miễn phí. có thể phản ánh chính sách freeship phổ biến kết hợp với một số trường hợp tính phí cao (giao hỏa tốc, đơn đặc biệt,...).


#### 1.2.9 Returns

In [22]:
print("The overall structure of data:\n")
df_returns.info()
print("\nThe random 5 lines of data:")
df_returns.sample(5)

The overall structure of data:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39939 entries, 0 to 39938
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   return_id        39939 non-null  object 
 1   order_id         39939 non-null  int64  
 2   product_id       39939 non-null  int64  
 3   return_date      39939 non-null  object 
 4   return_reason    39939 non-null  object 
 5   return_quantity  39939 non-null  int64  
 6   refund_amount    39939 non-null  float64
dtypes: float64(1), int64(3), object(3)
memory usage: 2.1+ MB

The random 5 lines of data:


,return_id,order_id,product_id,return_date,return_reason,return_quantity,refund_amount
32634,RET-042012,673391,596,2019-07-04,wrong_size,4,28080.14
29865,RET-038439,614136,1043,2018-08-05,wrong_size,4,13168.58
24488,RET-031494,500429,1062,2017-06-16,not_as_described,1,1774.84
20370,RET-026194,413726,952,2016-07-27,wrong_size,3,7760.17
31405,RET-040425,647278,1349,2019-02-11,defective,1,4274.71


In [23]:
duplicate_data = df_returns[df_returns.duplicated(keep=False)].sort_values(by=df_returns.columns.tolist())
if len(duplicate_data) > 0:
    print(f"{len(duplicate_data)} duplicated rows:")
    display(duplicate_data.head())

duplicate_primary_key = df_returns[df_returns['return_id'].duplicated(keep=False)].sort_values(by='return_id')
if len(duplicate_primary_key) > 0:
    print(f"{len(duplicate_primary_key)} duplicated return_id rows:")
    display(duplicate_primary_key.head())

print("\nDescriptive statistics of data:")
display(df_returns.describe())
display(df_returns.describe(include='O'))


Descriptive statistics of data:


,order_id,product_id,return_quantity,refund_amount
count,39939.000000,39939.000000,39939.000000,39939.000000
mean,409061.984176,1244.232730,2.743834,12784.458964
std,240063.904576,691.747822,1.828260,14092.150154
min,2.000000,3.000000,1.000000,458.810000
25%,202651.000000,702.000000,1.000000,3573.395000
50%,404254.000000,992.000000,2.000000,7888.880000
75%,615620.000000,2048.000000,4.000000,16881.990000
max,833351.000000,2412.000000,8.000000,160937.940000


,return_id,return_date,return_reason
count,39939,39939,39939
unique,39939,3806,5
top,RET-051504,2016-06-18,wrong_size
freq,1,34,13967


##### Phân tích tổng quan: 
- **Data structure:** 39939 rows and 7 columns.
- **Datatype:** 
    - `order_id` và `product_id` đang ở kiểu `int64`. Tuy nhiên, đây là các biến định danh và không có ý nghĩa toán học nên cần chuyển sang kiểu `string` để thuận tiện cho việc phân tích.
    - `return_id` đang ở kiểu `object`. Tuy nhiên, để thuận tiện cho việc phân tích và tối ưu bộ nhớ, chúng tôi sẽ chuyển sang kiểu `string`.
    - `return_reason` đang ở kiểu `object`. Tuy nhiên, số các giá trị unique trong các biến này tương đối ít so với số dòng trong toàn bộ dữ liệu nên cần chuyển sang kiểu `category` để thuận tiện trong việc phân loại.
    - `return_date` đang ở kiểu `object`. Tuy nhiên, đây là biến định danh ngày nên cần chuyển về kiểu `datetime`  để thuận tiện cho việc phân tích.
    - `refund_amount` đang ở kiểu `float64` -> phù hợp.
    - `return_quantity` đang ở kiểu `int64`-> phù hợp.
- **Missing value:** Không có Missing value trong toàn bộ dữ liệu.
- **Duplicate value:** Không có Duplicated value trong toàn bộ dữ liệu.
- **Descriptive statistics:** 
    - `return_quantity` có **mean > median** (~2.74 > 2) → phân phối lệch phải nhẹ.
    - `return_quantity` có khoảng giá trị **min ≈ 1** và **max ≈ 8**. Phân vị cho thấy giá trị chủ yếu nằm trong khoảng  **1 → 4**. Cho thấy phần lớn đơn trả hàng với số lượng nhỏ và vừa phải.
    
    - `refund_amount` có **mean > median** (~12 784 > ~7 889) → phân phối lệch phải rõ. Bên cạnh đó, độ lệch chuẩn lớn (~14 092) hơn mean cho thấy giá trị hoàn tiền phân tán mạnh.
    - `refund_amount` có khoảng giá trị rộng với **min ≈ 459** và **max ≈ 160 938**. Rất có thể tồn tại các outliers vì phân vị cho thấy **50%** đơn hàng nằm trong khoảng **(~3 573 → ~16 882)**. Như vậy, phần lớn đơn hàng có giá trị hoàn tiền ở mức trung bình, chỉ một số ít đơn hàng hoàn tiền lớn.

→ Nhìn chung, số lượng trả hàng thường nhỏ, nhưng giá trị hoàn tiền có thể biến động lớn.  Dữ liệu lệch phải, cần xử lý thêm **outliers** và cân nhắc **log transformation** để phân tích sâu hơn.

#### 1.2.10 Reviews

In [24]:
print("The overall structure of data:\n")
df_reviews.info()
print("\nThe random 5 lines of data:")
df_reviews.sample(5)

The overall structure of data:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113551 entries, 0 to 113550
Data columns (total 7 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   review_id     113551 non-null  object
 1   order_id      113551 non-null  int64 
 2   product_id    113551 non-null  int64 
 3   customer_id   113551 non-null  int64 
 4   review_date   113551 non-null  object
 5   rating        113551 non-null  int64 
 6   review_title  113551 non-null  object
dtypes: int64(4), object(3)
memory usage: 6.1+ MB

The random 5 lines of data:


,review_id,order_id,product_id,customer_id,review_date,rating,review_title
32123,REV-0041581,229487,1975,124387,2014-11-26,4,Good overall
37203,REV-0048116,264213,2396,17868,2015-04-25,3,"Decent, nothing special"
34842,REV-0045084,247588,791,139135,2015-02-18,5,Very satisfied
50802,REV-0065579,362489,2045,87371,2016-03-13,5,Excellent product!
96525,REV-0124764,699998,734,139059,2020-02-16,3,Average product


In [25]:
duplicate_data = df_reviews[df_reviews.duplicated(keep=False)].sort_values(by=df_reviews.columns.tolist())
if len(duplicate_data) > 0:
    print(f"{len(duplicate_data)} duplicated rows:")
    display(duplicate_data.head())

duplicate_primary_key = df_reviews[df_reviews['review_id'].duplicated(keep=False)].sort_values(by='review_id')
if len(duplicate_primary_key) > 0:
    print(f"{len(duplicate_primary_key)} duplicated review_id rows:")
    display(duplicate_primary_key.head())

print("\nDescriptive statistics of data:")
display(df_reviews.describe())
display(df_reviews.describe(include='O'))


Descriptive statistics of data:


,order_id,product_id,customer_id,rating
count,113551.000000,113551.000000,113551.000000,113551.000000
mean,408999.519740,1232.018705,85694.342762,3.936011
std,239021.922809,690.839232,48501.480918,1.149867
min,1.000000,3.000000,2.000000,1.000000
25%,202048.500000,689.000000,42096.000000,3.000000
50%,406841.000000,981.000000,89755.000000,4.000000
75%,614844.000000,2045.000000,133850.000000,5.000000
max,833296.000000,2412.000000,157563.000000,5.000000


,review_id,review_date,review_title
count,113551,113551,113551
unique,113551,3825,18
top,REV-0146984,2017-05-06,Very satisfied
freq,1,77,11450


##### Phân tích tổng quan: 
- **Data structure:** 113551 rows and 7 columns.
- **Datatype:** 
    - `order_id`, `product_id` và `customer_id` đang ở kiểu `int64`. Tuy nhiên, đây là các biến định danh và không có ý nghĩa toán học nên cần chuyển sang kiểu `string` để thuận tiện cho việc phân tích.
    - `review_id` đang ở kiểu `object`. Tuy nhiên, để thuận tiện cho việc phân tích và tối ưu bộ nhớ, chúng tôi sẽ chuyển sang kiểu `string`.
    - `review_title` đang ở kiểu `object`. Tuy nhiên, số các giá trị unique trong các biến này tương đối ít so với số dòng trong toàn bộ dữ liệu nên cần chuyển sang kiểu `category` để thuận tiện trong việc phân loại.
    - `review_date` đang ở kiểu `object`. Tuy nhiên, đây là biến định danh ngày nên cần chuyển về kiểu `datetime`  để thuận tiện cho việc phân tích.
    - `rating` đang ở kiểu `int64`. Tuy nhiên với min = 1 và max = 5, rất có thể đây cũng là biến phân loại -> cần kiểm tra lại để cân nhắc chuyển sang kiểu `category`.
- **Missing value:** Không có Missing value trong toàn bộ dữ liệu.
- **Duplicate value:** Không có Duplicated value trong toàn bộ dữ liệu.

In [26]:
print(df_reviews['rating'].value_counts())

rating
5    45256
4    36412
3    17016
2     9095
1     5772
Name: count, dtype: int64


Kết quả cho thấy `rating` chỉ tồn tại 5 giá trị distinct -> là một biến phân loại như mong đợi, cần chuyển sang kiểu `category`

#### 1.2.11 Sale daily revenue data

In [27]:
print("The overall structure of data:\n")
df_sales.info()
print("\nThe random 5 lines of data:")
df_sales.sample(5)

The overall structure of data:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3833 entries, 0 to 3832
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Date     3833 non-null   object 
 1   Revenue  3833 non-null   float64
 2   COGS     3833 non-null   float64
dtypes: float64(2), object(1)
memory usage: 90.0+ KB

The random 5 lines of data:


,Date,Revenue,COGS
607,2014-03-03,3637156.46,2881941.03
2944,2020-07-26,2852671.37,2228931.73
2379,2019-01-08,1715399.79,1347590.83
2459,2019-03-29,4309156.23,3848197.98
3381,2021-10-06,1793088.31,1465294.60


In [28]:
duplicate_data = df_sales[df_sales.duplicated(keep=False)].sort_values(by=df_sales.columns.tolist())
if len(duplicate_data) > 0:
    print(f"{len(duplicate_data)} duplicated rows:")
    display(duplicate_data.head())

duplicate_primary_key = df_sales[df_sales['Date'].duplicated(keep=False)].sort_values(by='Date')
if len(duplicate_primary_key) > 0:
    print(f"{len(duplicate_primary_key)} duplicated Date rows:")
    display(duplicate_primary_key.head())

print("\nDescriptive statistics of data:")
display(df_sales.describe())
display(df_sales.describe(include='O'))


Descriptive statistics of data:


,Revenue,COGS
count,3.833000e+03,3.833000e+03
mean,4.286584e+06,3.695134e+06
std,2.624840e+06,2.219789e+06
min,2.798139e+05,2.365763e+05
25%,2.471089e+06,2.150580e+06
50%,3.647304e+06,3.161113e+06
75%,5.350877e+06,4.637294e+06
max,2.090527e+07,1.653586e+07


,Date
count,3833
unique,3833
top,2022-12-31
freq,1


##### Phân tích tổng quan: 
- **Data structure:** 3833 rows and 3 columns.
- **Datatype:** 
    - `Date` đang ở kiểu `object`. Tuy nhiên, đây là biến định danh ngày nên cần chuyển về kiểu `datetime`  để thuận tiện cho việc phân tích.
    - `Revenue` và `COGS` đang ở kiểu `float64` -> phù hợp.
- **Missing value:** Không có Missing value trong toàn bộ dữ liệu.
- **Duplicate value:** Không có Duplicated value trong toàn bộ dữ liệu.

#### 1.2.12 Inventory

In [29]:
print("The overall structure of data:\n")
df_inventory.info()
print("\nThe random 5 lines of data:")
df_inventory.sample(5)

The overall structure of data:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60247 entries, 0 to 60246
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   snapshot_date      60247 non-null  object 
 1   product_id         60247 non-null  int64  
 2   stock_on_hand      60247 non-null  int64  
 3   units_received     60247 non-null  int64  
 4   units_sold         60247 non-null  int64  
 5   stockout_days      60247 non-null  int64  
 6   days_of_supply     60247 non-null  float64
 7   fill_rate          60247 non-null  float64
 8   stockout_flag      60247 non-null  int64  
 9   overstock_flag     60247 non-null  int64  
 10  reorder_flag       60247 non-null  int64  
 11  sell_through_rate  60247 non-null  float64
 12  product_name       60247 non-null  object 
 13  category           60247 non-null  object 
 14  segment            60247 non-null  object 
 15  year               60247 non-null  int

,snapshot_date,product_id,stock_on_hand,units_received,units_sold,stockout_days,days_of_supply,fill_rate,stockout_flag,overstock_flag,reorder_flag,sell_through_rate,product_name,category,segment,year,month
38251,2019-02-28,1605,3,1,1,0,90.0,1.0000,0,0,0,0.2500,VietMode RS-43,Outdoor,Premium,2019,2
35630,2013-07-31,1492,13,2,2,0,195.0,1.0000,0,1,0,0.1333,VietMode RP-84,Outdoor,Activewear,2013,7
43102,2017-05-31,1884,99,16,13,2,228.5,0.9333,1,1,0,0.1161,BambooCraft UC-03,Streetwear,Everyday,2017,5
6615,2015-06-30,475,1007,262,238,0,126.9,1.0000,0,1,0,0.1912,SaigonFlex UM-80,Streetwear,Balanced,2015,6
50238,2020-11-30,2098,279,4,4,0,2092.5,1.0000,0,1,0,0.0141,UrbanVN YY-02,GenZ,Trendy,2020,11


In [30]:
duplicate_data = df_inventory[df_inventory.duplicated(keep=False)].sort_values(by=df_inventory.columns.tolist())
if len(duplicate_data) > 0:
    print(f"{len(duplicate_data)} duplicated rows:")
    display(duplicate_data.head())

print("\nDescriptive statistics of data:")
display(df_inventory.describe())
display(df_inventory.describe(include='O'))


Descriptive statistics of data:


,product_id,stock_on_hand,units_received,units_sold,stockout_days,days_of_supply,fill_rate,stockout_flag,overstock_flag,reorder_flag,sell_through_rate,year,month
count,60247.000000,60247.000000,60247.000000,60247.000000,60247.000000,60247.000000,60247.000000,60247.000000,60247.000000,60247.0,60247.000000,60247.000000,60247.000000
mean,1311.408468,189.298455,18.046807,15.417764,1.160639,912.677576,0.961312,0.673411,0.762561,0.0,0.152275,2017.222799,6.617292
std,673.051769,316.976124,34.080228,28.404379,1.624490,2587.624108,0.054156,0.468969,0.425517,0.0,0.139291,2.972353,3.385629
min,1.000000,3.000000,1.000000,1.000000,0.000000,5.200000,0.066700,0.000000,0.000000,0.0,0.000400,2012.000000,1.000000
25%,760.000000,15.000000,2.000000,2.000000,0.000000,96.000000,0.933300,0.000000,1.000000,0.0,0.042100,2015.000000,4.000000
50%,1223.000000,62.000000,6.000000,6.000000,1.000000,240.000000,0.966700,1.000000,1.000000,0.0,0.111100,2017.000000,7.000000
75%,1942.000000,210.000000,19.000000,16.000000,2.000000,683.100000,1.000000,1.000000,1.000000,0.0,0.238100,2020.000000,10.000000
max,2412.000000,2673.000000,817.000000,670.000000,28.000000,68100.000000,1.000000,1.000000,1.000000,0.0,0.853100,2022.000000,12.000000


,snapshot_date,product_name,category,segment
count,60247,60247,60247,60247
unique,126,1465,4,8
top,2018-05-31,VietMode RP-10,Streetwear,Activewear
freq,562,203,31020,18290


##### Phân tích tổng quan: 
- **Data structure:** 60247 rows and 17 columns.
- **Datatype:** 
    - `product_id` đang ở kiểu `int64`. Tuy nhiên, đây là các biến định danh và không có ý nghĩa toán học nên cần chuyển sang kiểu `string` để thuận tiện cho việc phân tích.
    - `product_name` đang ở kiểu `object`. Tuy nhiên, để thuận tiện cho việc phân tích và tối ưu bộ nhớ, chúng tôi sẽ chuyển sang kiểu `string`.
    - `category` và `segment` đang ở kiểu `object`. Tuy nhiên, số các giá trị unique trong các biến này tương đối ít so với số dòng trong toàn bộ dữ liệu nên cần chuyển sang kiểu `category` để thuận tiện trong việc phân loại.
    - `snapshot_date` đang ở kiểu `object`. Tuy nhiên, đây là biến định danh ngày nên cần chuyển về kiểu `datetime`  để thuận tiện cho việc phân tích.
    - `days_of_supply`, `fill_rate` và `sell_through_rate` đang ở kiểu `float64` -> phù hợp.
    - `stock_on_hand`, `units_received`, `units_sold`, `stockout_days`, `month` và `year` đang ở kiểu `int64` -> phù hợp.
    - `stockout_flag`, `overstock_flag` và `reorder_flag` đang ở kiểu `int64`. Tuy nhiên với min = 0 và max = 1 (hoặc 0), rất có thể đây cũng là biến phân loại -> cần kiểm tra lại để cân nhắc chuyển sang kiểu phù hợp.
- **Missing value:** Không có Missing value trong toàn bộ dữ liệu.
- **Duplicate value:** Không có Duplicated value trong toàn bộ dữ liệu.

In [31]:
print(df_inventory['stockout_flag'].value_counts())
print(df_inventory['overstock_flag'].value_counts())
print(df_inventory['reorder_flag'].value_counts())

stockout_flag
1    40571
0    19676
Name: count, dtype: int64
overstock_flag
1    45942
0    14305
Name: count, dtype: int64
reorder_flag
0    60247
Name: count, dtype: int64


- Kết quả cho thấy `stockout_flag`, `overstock_flag` và `reorder_flag` chỉ tồn tại 2 giá trị 0 và 1 -> là một biến phân loại như mong đợi, cần chuyển sang kiểu `bool`.

#### 1.2.13 Web traffic

In [32]:
print("The overall structure of data:\n")
df_web_traffic.info()
print("\nThe random 5 lines of data:")
df_web_traffic.sample(5)

The overall structure of data:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3652 entries, 0 to 3651
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   date                      3652 non-null   object 
 1   sessions                  3652 non-null   int64  
 2   unique_visitors           3652 non-null   int64  
 3   page_views                3652 non-null   int64  
 4   bounce_rate               3652 non-null   float64
 5   avg_session_duration_sec  3652 non-null   float64
 6   traffic_source            3652 non-null   object 
dtypes: float64(2), int64(3), object(2)
memory usage: 199.8+ KB

The random 5 lines of data:


,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
3346,2022-03-01,35076,24922,188403,0.00451,128.5,paid_search
3193,2021-09-29,29661,23727,120596,0.00507,281.8,email_campaign
1795,2017-12-01,16368,12107,53712,0.00464,256.7,organic_search
1185,2016-03-31,27301,21181,141142,0.00560,102.6,paid_search
2088,2018-09-20,25444,19205,132990,0.00518,184.4,social_media


In [33]:
duplicate_data = df_web_traffic[df_web_traffic.duplicated(keep=False)].sort_values(by=df_web_traffic.columns.tolist())
if len(duplicate_data) > 0:
    print(f"{len(duplicate_data)} duplicated rows:")
    display(duplicate_data.head())

duplicate_primary_key = df_web_traffic[df_web_traffic['date'].duplicated(keep=False)].sort_values(by='date')
if len(duplicate_primary_key) > 0:
    print(f"{len(duplicate_primary_key)} duplicated date rows:")
    display(duplicate_primary_key.head())

print("\nDescriptive statistics of data:")
display(df_web_traffic.describe())
display(df_web_traffic.describe(include='O'))


Descriptive statistics of data:


,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec
count,3652.000000,3652.000000,3652.000000,3652.000000,3652.000000
mean,25041.768072,19031.404436,108615.224535,0.004487,210.283242
std,9422.609335,7237.953062,44472.055524,0.000753,63.771711
min,7973.000000,6136.000000,30451.000000,0.003200,100.100000
25%,17099.250000,12915.000000,72982.000000,0.003848,156.700000
50%,23633.500000,17924.000000,101010.500000,0.004450,209.200000
75%,31782.750000,24191.750000,138086.000000,0.005160,266.200000
max,50947.000000,40430.000000,275560.000000,0.005800,319.900000


,date,traffic_source
count,3652,3652
unique,3652,6
top,2022-12-31,organic_search
freq,1,1090


##### Phân tích tổng quan: 
- **Data structure:** 3652 rows and 7 columns.
- **Datatype:** 
    - `traffic_source` đang ở kiểu `object`. Tuy nhiên, số các giá trị unique trong các biến này tương đối ít so với số dòng trong toàn bộ dữ liệu nên cần chuyển sang kiểu `category` để thuận tiện trong việc phân loại.
    - `date` đang ở kiểu `object`. Tuy nhiên, đây là biến định danh ngày nên cần chuyển về kiểu `datetime`  để thuận tiện cho việc phân tích.
    - `bounce_rate` và `avg_session_duration_sec` đang ở kiểu `float64` -> phù hợp.
    - `sessions`, `unique_visitors` và `page_views` đang ở kiểu `int64` -> phù hợp.
- **Missing value:** Không có Missing value trong toàn bộ dữ liệu.
- **Duplicate value:** Không có Duplicated value trong toàn bộ dữ liệu.

## Phase 2: Data Cleaning
Bước đầu xử lý dữ liệu dựa trên các mô tả và phân tích ở Phase 1.

### 2.1 Xử lý kiểu dữ liệu:

#### 2.1.1 Product categories

In [34]:
cols_to_string = ['product_id', 'product_name']
cols_to_category = ['category', 'segment', 'size', 'color']

df_products[cols_to_string] = df_products[cols_to_string].astype('string')
df_products[cols_to_category] = df_products[cols_to_category].astype('category')

print("Display structure of data after data type casting: \n")
df_products.info()

Display structure of data after data type casting: 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2412 entries, 0 to 2411
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   product_id    2412 non-null   string  
 1   product_name  2412 non-null   string  
 2   category      2412 non-null   category
 3   segment       2412 non-null   category
 4   size          2412 non-null   category
 5   color         2412 non-null   category
 6   price         2412 non-null   float64 
 7   cogs          2412 non-null   float64 
dtypes: category(4), float64(2), string(2)
memory usage: 86.0 KB


Kiểu dữ liệu của các biến hiện tại đã phù hợp. Tuy nhiên, cần kiểm tra các giá trị unique trong các biến này để đảm bảo không có giá trị trùng lặp ý nghĩa hoặc các giá trị không mang ý nghĩa.

In [35]:
for val in cols_to_category:
    print(f"Total unique values of '{val}' column: {df_products[val].nunique()}")
    counts = df_products[val].value_counts()
    percents = df_products[val].value_counts(normalize=True) * 100
    summary_df = pd.DataFrame({
        'Count': counts,
        'Percentage': percents.apply(lambda x: f"{x:.2f}%".replace('.', ','))
    })
    print(summary_df.to_string())
    print("-" * 40, "\n")

Total unique values of 'category' column: 4
            Count Percentage
category                    
Streetwear   1320     54,73%
Outdoor       743     30,80%
Casual        201      8,33%
GenZ          148      6,14%
---------------------------------------- 

Total unique values of 'segment' column: 8
             Count Percentage
segment                      
Activewear     598     24,79%
Everyday       405     16,79%
Performance    347     14,39%
Balanced       306     12,69%
Standard       262     10,86%
Premium        177      7,34%
All-weather    169      7,01%
Trendy         148      6,14%
---------------------------------------- 

Total unique values of 'size' column: 4
      Count Percentage
size                  
L       603     25,00%
M       603     25,00%
S       603     25,00%
XL      603     25,00%
---------------------------------------- 

Total unique values of 'color' column: 10
        Count Percentage
color                   
black     242     10,03%
orange    242  

Hiện tại, các giá trị unique trong các biến đều có vẻ hợp lý và mang ý nghĩa. Không cần chỉnh sửa thêm

#### 2.1.2 Customers

In [36]:
cols_to_string = ['customer_id', 'zip']
cols_to_category = ['gender', 'age_group', 'city', 'acquisition_channel']
cols_to_datetime = ['signup_date']

df_customers[cols_to_string] = df_customers[cols_to_string].astype('string')
df_customers[cols_to_category] = df_customers[cols_to_category].astype('category')

for col in cols_to_datetime:
    df_customers[col] = pd.to_datetime(df_customers[col], format='%Y-%m-%d', errors='coerce')

print("Display structure of data after data type casting: \n")
df_customers.info()

Display structure of data after data type casting: 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121930 entries, 0 to 121929
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   customer_id          121930 non-null  string        
 1   zip                  121930 non-null  string        
 2   city                 121930 non-null  category      
 3   signup_date          121930 non-null  datetime64[ns]
 4   gender               121930 non-null  category      
 5   age_group            121930 non-null  category      
 6   acquisition_channel  121930 non-null  category      
dtypes: category(4), datetime64[ns](1), string(2)
memory usage: 3.3 MB


Kiểu dữ liệu của các biến hiện tại đã phù hợp. Tuy nhiên, cần kiểm tra các giá trị unique trong các biến này để đảm bảo không có giá trị trùng lặp ý nghĩa hoặc các giá trị không mang ý nghĩa.

In [37]:
for val in cols_to_category:
    print(f"Total unique values of '{val}' column: {df_customers[val].nunique()}")
    counts = df_customers[val].value_counts()
    percents = df_customers[val].value_counts(normalize=True) * 100
    summary_df = pd.DataFrame({
        'Count': counts,
        'Percentage': percents.apply(lambda x: f"{x:.2f}%".replace('.', ','))
    })
    print(summary_df.to_string())
    print("-" * 40, "\n")

Total unique values of 'gender' column: 3
            Count Percentage
gender                      
Female      59640     48,91%
Male        57457     47,12%
Non-binary   4833      3,96%
---------------------------------------- 

Total unique values of 'age_group' column: 5
           Count Percentage
age_group                  
25-34      36342     29,81%
35-44      31920     26,18%
45-54      23172     19,00%
18-24      17039     13,97%
55+        13457     11,04%
---------------------------------------- 

Total unique values of 'city' column: 42
                     Count Percentage
city                                 
Cam Pha               4398      3,61%
Thai Nguyen           4347      3,57%
Phu Ly                4243      3,48%
Hanoi                 4240      3,48%
Ha Long               4236      3,47%
Bac Ninh              4172      3,42%
Hai Phong             4170      3,42%
Nam Dinh              4169      3,42%
Bac Giang             4160      3,41%
Ninh Binh             4081 

Hiện tại, các giá trị unique trong các biến đều có vẻ hợp lý và mang ý nghĩa. Không cần chỉnh sửa thêm

#### 2.1.3 Promotional programs

In [38]:
cols_to_string = ['promo_id', 'promo_name']
cols_to_category = ['promo_type', 'applicable_category', 'promo_channel']
cols_to_bool = ['stackable_flag']
cols_to_datetime = ['start_date', 'end_date']

df_promotions[cols_to_string] = df_promotions[cols_to_string].astype('string')
df_promotions[cols_to_category] = df_promotions[cols_to_category].astype('category')
df_promotions[cols_to_bool] = df_promotions[cols_to_bool].astype(bool)

for col in cols_to_datetime:
    df_promotions[col] = pd.to_datetime(df_promotions[col], format='%Y-%m-%d', errors='coerce')

print("Display structure of data after data type casting: \n")
df_promotions.info()

Display structure of data after data type casting: 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   promo_id             50 non-null     string        
 1   promo_name           50 non-null     string        
 2   promo_type           50 non-null     category      
 3   discount_value       50 non-null     float64       
 4   start_date           50 non-null     datetime64[ns]
 5   end_date             50 non-null     datetime64[ns]
 6   applicable_category  10 non-null     category      
 7   promo_channel        50 non-null     category      
 8   stackable_flag       50 non-null     bool          
 9   min_order_value      50 non-null     int64         
dtypes: bool(1), category(3), datetime64[ns](2), float64(1), int64(1), string(2)
memory usage: 3.1 KB


Kiểu dữ liệu của các biến hiện tại đã phù hợp. Tuy nhiên, cần kiểm tra các giá trị unique trong các biến này để đảm bảo không có giá trị trùng lặp ý nghĩa hoặc các giá trị không mang ý nghĩa.

In [39]:
for val in cols_to_category:
    print(f"Total unique values of '{val}' column: {df_promotions[val].nunique()}")
    counts = df_promotions[val].value_counts()
    percents = df_promotions[val].value_counts(normalize=True) * 100
    summary_df = pd.DataFrame({
        'Count': counts,
        'Percentage': percents.apply(lambda x: f"{x:.2f}%".replace('.', ','))
    })
    print(summary_df.to_string())
    print("-" * 40, "\n")

Total unique values of 'promo_type' column: 2
            Count Percentage
promo_type                  
percentage     45     90,00%
fixed           5     10,00%
---------------------------------------- 

Total unique values of 'applicable_category' column: 2
                     Count Percentage
applicable_category                  
Outdoor                  5     50,00%
Streetwear               5     50,00%
---------------------------------------- 

Total unique values of 'promo_channel' column: 5
               Count Percentage
promo_channel                  
all_channels      19     38,00%
online            13     26,00%
email              7     14,00%
social_media       6     12,00%
in_store           5     10,00%
---------------------------------------- 



Hiện tại, các giá trị unique trong các biến đều có vẻ hợp lý và mang ý nghĩa. Không cần chỉnh sửa thêm

#### 2.1.4 Geography

In [40]:
cols_to_string = ['zip']
cols_to_category = ['city', 'region', 'district']

df_geography[cols_to_string] = df_geography[cols_to_string].astype('string')
df_geography[cols_to_category] = df_geography[cols_to_category].astype('category')

print("Display structure of data after data type casting: \n")
df_geography.info()

Display structure of data after data type casting: 



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39948 entries, 0 to 39947
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   zip       39948 non-null  string  
 1   city      39948 non-null  category
 2   region    39948 non-null  category
 3   district  39948 non-null  category
dtypes: category(3), string(1)
memory usage: 432.1 KB


Kiểu dữ liệu của các biến hiện tại đã phù hợp. Tuy nhiên, cần kiểm tra các giá trị unique trong các biến này để đảm bảo không có giá trị trùng lặp ý nghĩa hoặc các giá trị không mang ý nghĩa.

In [41]:
for val in cols_to_category:
    print(f"Total unique values of '{val}' column: {df_geography[val].nunique()}")
    counts = df_geography[val].value_counts()
    percents = df_geography[val].value_counts(normalize=True) * 100
    summary_df = pd.DataFrame({
        'Count': counts,
        'Percentage': percents.apply(lambda x: f"{x:.2f}%".replace('.', ','))
    })
    print(summary_df.to_string())
    print("-" * 40, "\n")

Total unique values of 'city' column: 42
                     Count Percentage
city                                 
Cam Pha               1403      3,51%
Phu Ly                1399      3,50%
Thai Nguyen           1394      3,49%
Hanoi                 1376      3,44%
Nam Dinh              1370      3,43%
Ha Long               1357      3,40%
Bac Giang             1347      3,37%
Bac Ninh              1346      3,37%
Hai Phong             1346      3,37%
Son Tay               1344      3,36%
Ninh Binh             1326      3,32%
Uong Bi               1325      3,32%
Viet Tri              1324      3,31%
Lao Cai               1272      3,18%
Kon Tum               1265      3,17%
Hoi An                1255      3,14%
Dong Hoi              1246      3,12%
Phan Rang-Thap Cham   1219      3,05%
Hue                   1216      3,04%
Tuy Hoa               1212      3,03%
Nha Trang             1199      3,00%
Quang Ngai            1192      2,98%
Phan Thiet            1189      2,98%
Tam Ky   

Hiện tại, các giá trị unique trong các biến đều có vẻ hợp lý và mang ý nghĩa. Không cần chỉnh sửa thêm

#### 2.1.5 Orders

In [42]:
cols_to_string = ['order_id', 'customer_id', 'zip']
cols_to_category = ['order_status', 'payment_method', 'device_type', 'order_source']
cols_to_datetime = ['order_date']

df_orders[cols_to_string] = df_orders[cols_to_string].astype('string')
df_orders[cols_to_category] = df_orders[cols_to_category].astype('category')

for col in cols_to_datetime:
    df_orders[col] = pd.to_datetime(df_orders[col], format='%Y-%m-%d', errors='coerce')

print("Display structure of data after data type casting: \n")
df_orders.info()

Display structure of data after data type casting: 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   order_id        646945 non-null  string        
 1   order_date      646945 non-null  datetime64[ns]
 2   customer_id     646945 non-null  string        
 3   zip             646945 non-null  string        
 4   order_status    646945 non-null  category      
 5   payment_method  646945 non-null  category      
 6   device_type     646945 non-null  category      
 7   order_source    646945 non-null  category      
dtypes: category(4), datetime64[ns](1), string(3)
memory usage: 22.2 MB


Kiểu dữ liệu của các biến hiện tại đã phù hợp. Tuy nhiên, cần kiểm tra các giá trị unique trong các biến này để đảm bảo không có giá trị trùng lặp ý nghĩa hoặc các giá trị không mang ý nghĩa.

In [43]:
for val in cols_to_category:
    print(f"Total unique values of '{val}' column: {df_orders[val].nunique()}")
    counts = df_orders[val].value_counts()
    percents = df_orders[val].value_counts(normalize=True) * 100
    summary_df = pd.DataFrame({
        'Count': counts,
        'Percentage': percents.apply(lambda x: f"{x:.2f}%".replace('.', ','))
    })
    print(summary_df.to_string())
    print("-" * 40, "\n")

Total unique values of 'order_status' column: 6
               Count Percentage
order_status                   
delivered     516716     79,87%
cancelled      59462      9,19%
returned       36142      5,59%
shipped        13773      2,13%
paid           13577      2,10%
created         7275      1,12%
---------------------------------------- 

Total unique values of 'payment_method' column: 5
                 Count Percentage
payment_method                   
credit_card     356352     55,08%
paypal           97018     15,00%
cod              96681     14,94%
apple_pay        64763     10,01%
bank_transfer    32131      4,97%
---------------------------------------- 

Total unique values of 'device_type' column: 3
              Count Percentage
device_type                   
mobile       291482     45,06%
desktop      258855     40,01%
tablet        96608     14,93%
---------------------------------------- 

Total unique values of 'order_source' column: 6
                 Count Percen

Hiện tại, các giá trị unique trong các biến đều có vẻ hợp lý và mang ý nghĩa. Không cần chỉnh sửa thêm

#### 2.1.6 Order detail items

In [44]:
cols_to_string = ['order_id', 'product_id', 'promo_id', 'promo_id_2']

df_order_items[cols_to_string] = df_order_items[cols_to_string].astype('string')

print("Display structure of data after data type casting: \n")
df_order_items.info()

Display structure of data after data type casting: 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 714669 entries, 0 to 714668
Data columns (total 7 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   order_id         714669 non-null  string 
 1   product_id       714669 non-null  string 
 2   quantity         714669 non-null  int64  
 3   unit_price       714669 non-null  float64
 4   discount_amount  714669 non-null  float64
 5   promo_id         276316 non-null  string 
 6   promo_id_2       206 non-null     string 
dtypes: float64(2), int64(1), string(4)
memory usage: 38.2 MB


Kiểu dữ liệu của các biến hiện tại đã phù hợp. 

#### 2.1.7 Payments

In [45]:
df_payments['installments'] = df_payments['installments'].apply(lambda x: f"{x} period" if x == 1 else f"{x} periods")

cols_to_string = ['order_id']
cols_to_category = ['payment_method', 'installments']

df_payments[cols_to_string] = df_payments[cols_to_string].astype('string')
df_payments[cols_to_category] = df_payments[cols_to_category].astype('category')

print("Display structure of data after data type casting: \n")
df_payments.info()

Display structure of data after data type casting: 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 4 columns):
 #   Column          Non-Null Count   Dtype   
---  ------          --------------   -----   
 0   order_id        646945 non-null  string  
 1   payment_method  646945 non-null  category
 2   payment_value   646945 non-null  float64 
 3   installments    646945 non-null  category
dtypes: category(2), float64(1), string(1)
memory usage: 11.1 MB


Kiểu dữ liệu của các biến hiện tại đã phù hợp. Tuy nhiên, cần kiểm tra các giá trị unique trong các biến này để đảm bảo không có giá trị trùng lặp ý nghĩa hoặc các giá trị không mang ý nghĩa.

In [46]:
for val in cols_to_category:
    print(f"Total unique values of '{val}' column: {df_payments[val].nunique()}")
    counts = df_payments[val].value_counts()
    percents = df_payments[val].value_counts(normalize=True) * 100
    summary_df = pd.DataFrame({
        'Count': counts,
        'Percentage': percents.apply(lambda x: f"{x:.2f}%".replace('.', ','))
    })
    print(summary_df.to_string())
    print("-" * 40, "\n")

Total unique values of 'payment_method' column: 5
                 Count Percentage
payment_method                   
credit_card     356352     55,08%
paypal           97018     15,00%
cod              96681     14,94%
apple_pay        64763     10,01%
bank_transfer    32131      4,97%
---------------------------------------- 

Total unique values of 'installments' column: 5
               Count Percentage
installments                   
1 period      262866     40,63%
3 periods     218949     33,84%
6 periods     109910     16,99%
12 periods     54126      8,37%
2 periods       1094      0,17%
---------------------------------------- 



Hiện tại, các giá trị unique trong các biến đều có vẻ hợp lý và mang ý nghĩa. Không cần chỉnh sửa thêm

#### 2.1.8 Shipments

In [47]:
cols_to_string = ['order_id']
cols_to_datetime = ['ship_date', 'delivery_date']

df_shipments[cols_to_string] = df_shipments[cols_to_string].astype('string')

for col in cols_to_datetime:
    df_shipments[col] = pd.to_datetime(df_shipments[col], format='%Y-%m-%d', errors='coerce')

print("Display structure of data after data type casting: \n")
df_shipments.info()

Display structure of data after data type casting: 



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 566067 entries, 0 to 566066
Data columns (total 4 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   order_id       566067 non-null  string        
 1   ship_date      566067 non-null  datetime64[ns]
 2   delivery_date  566067 non-null  datetime64[ns]
 3   shipping_fee   566067 non-null  float64       
dtypes: datetime64[ns](2), float64(1), string(1)
memory usage: 17.3 MB


Kiểu dữ liệu của các biến hiện tại đã phù hợp. 

#### 2.1.9 Returns

In [48]:
cols_to_string = ['order_id', 'product_id', 'return_id']
cols_to_category = ['return_reason']
cols_to_datetime = ['return_date']

df_returns[cols_to_string] = df_returns[cols_to_string].astype('string')
df_returns[cols_to_category] = df_returns[cols_to_category].astype('category')

for col in cols_to_datetime:
    df_returns[col] = pd.to_datetime(df_returns[col], format='%Y-%m-%d', errors='coerce')

print("Display structure of data after data type casting: \n")
df_returns.info()

Display structure of data after data type casting: 



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39939 entries, 0 to 39938
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   return_id        39939 non-null  string        
 1   order_id         39939 non-null  string        
 2   product_id       39939 non-null  string        
 3   return_date      39939 non-null  datetime64[ns]
 4   return_reason    39939 non-null  category      
 5   return_quantity  39939 non-null  int64         
 6   refund_amount    39939 non-null  float64       
dtypes: category(1), datetime64[ns](1), float64(1), int64(1), string(3)
memory usage: 1.9 MB


Kiểu dữ liệu của các biến hiện tại đã phù hợp. Tuy nhiên, cần kiểm tra các giá trị unique trong các biến này để đảm bảo không có giá trị trùng lặp ý nghĩa hoặc các giá trị không mang ý nghĩa.

In [49]:
for val in cols_to_category:
    print(f"Total unique values of '{val}' column: {df_returns[val].nunique()}")
    counts = df_returns[val].value_counts()
    percents = df_returns[val].value_counts(normalize=True) * 100
    summary_df = pd.DataFrame({
        'Count': counts,
        'Percentage': percents.apply(lambda x: f"{x:.2f}%".replace('.', ','))
    })
    print(summary_df.to_string())
    print("-" * 40, "\n")

Total unique values of 'return_reason' column: 5
                  Count Percentage
return_reason                     
wrong_size        13967     34,97%
defective          8020     20,08%
not_as_described   7035     17,61%
changed_mind       6931     17,35%
late_delivery      3986      9,98%
---------------------------------------- 



Hiện tại, các giá trị unique trong các biến đều có vẻ hợp lý và mang ý nghĩa. Không cần chỉnh sửa thêm

#### 2.1.10 Reviews

In [50]:
df_reviews['rating'] = df_reviews['rating'].apply(lambda x: f"{x}*")

cols_to_string = ['order_id', 'product_id', 'customer_id', 'review_id']
cols_to_category = ['review_title', 'rating']
cols_to_datetime = ['review_date']

df_reviews[cols_to_string] = df_reviews[cols_to_string].astype('string')
df_reviews[cols_to_category] = df_reviews[cols_to_category].astype('category')

for col in cols_to_datetime:
    df_reviews[col] = pd.to_datetime(df_reviews[col], format='%Y-%m-%d', errors='coerce')

print("Display structure of data after data type casting: \n")
df_reviews.info()

Display structure of data after data type casting: 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113551 entries, 0 to 113550
Data columns (total 7 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   review_id     113551 non-null  string        
 1   order_id      113551 non-null  string        
 2   product_id    113551 non-null  string        
 3   customer_id   113551 non-null  string        
 4   review_date   113551 non-null  datetime64[ns]
 5   rating        113551 non-null  category      
 6   review_title  113551 non-null  category      
dtypes: category(2), datetime64[ns](1), string(4)
memory usage: 4.5 MB


Kiểu dữ liệu của các biến hiện tại đã phù hợp. Tuy nhiên, cần kiểm tra các giá trị unique trong các biến này để đảm bảo không có giá trị trùng lặp ý nghĩa hoặc các giá trị không mang ý nghĩa.

In [51]:
for val in cols_to_category:
    print(f"Total unique values of '{val}' column: {df_reviews[val].nunique()}")
    counts = df_reviews[val].value_counts()
    percents = df_reviews[val].value_counts(normalize=True) * 100
    summary_df = pd.DataFrame({
        'Count': counts,
        'Percentage': percents.apply(lambda x: f"{x:.2f}%".replace('.', ','))
    })
    print(summary_df.to_string())
    print("-" * 40, "\n")

Total unique values of 'review_title' column: 18
                         Count Percentage
review_title                             
Very satisfied           11450     10,08%
Highly recommend         11407     10,05%
Great quality            11218      9,88%
Excellent product!       11181      9,85%
Good overall              9185      8,09%
Happy with purchase       9171      8,08%
Solid choice              9070      7,99%
Works well                8986      7,91%
Mixed feelings            5706      5,03%
Average product           5656      4,98%
Decent, nothing special   5654      4,98%
Some issues               3037      2,67%
Would not reorder         3034      2,67%
Below expectations        3024      2,66%
Would not recommend       1460      1,29%
Poor quality              1443      1,27%
Very disappointed         1442      1,27%
Not as described          1427      1,26%
---------------------------------------- 

Total unique values of 'rating' column: 5
        Count Percentage
r

Hiện tại, các giá trị unique trong các biến đều có vẻ hợp lý và mang ý nghĩa. Không cần chỉnh sửa thêm

#### 2.1.11 Sale daily revenue data

In [52]:
cols_to_datetime = ['Date']

for col in cols_to_datetime:
    df_sales[col] = pd.to_datetime(df_sales[col], format='%Y-%m-%d', errors='coerce')

print("Display structure of data after data type casting: \n")
df_sales.info()

Display structure of data after data type casting: 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3833 entries, 0 to 3832
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   Date     3833 non-null   datetime64[ns]
 1   Revenue  3833 non-null   float64       
 2   COGS     3833 non-null   float64       
dtypes: datetime64[ns](1), float64(2)
memory usage: 90.0 KB


Kiểu dữ liệu của các biến hiện tại đã phù hợp. 

#### 2.1.12 Inventory

In [53]:
cols_to_string = ['product_id', 'product_name']
cols_to_category = ['category', 'segment']
cols_to_bool = ['stockout_flag', 'overstock_flag', 'reorder_flag']
cols_to_datetime = ['snapshot_date']

df_inventory[cols_to_string] = df_inventory[cols_to_string].astype('string')
df_inventory[cols_to_category] = df_inventory[cols_to_category].astype('category')
df_inventory[cols_to_bool] = df_inventory[cols_to_bool].astype(bool)

for col in cols_to_datetime:
    df_inventory[col] = pd.to_datetime(df_inventory[col], format='%Y-%m-%d', errors='coerce')

print("Display structure of data after data type casting: \n")
df_inventory.info()

Display structure of data after data type casting: 



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60247 entries, 0 to 60246
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   snapshot_date      60247 non-null  datetime64[ns]
 1   product_id         60247 non-null  string        
 2   stock_on_hand      60247 non-null  int64         
 3   units_received     60247 non-null  int64         
 4   units_sold         60247 non-null  int64         
 5   stockout_days      60247 non-null  int64         
 6   days_of_supply     60247 non-null  float64       
 7   fill_rate          60247 non-null  float64       
 8   stockout_flag      60247 non-null  bool          
 9   overstock_flag     60247 non-null  bool          
 10  reorder_flag       60247 non-null  bool          
 11  sell_through_rate  60247 non-null  float64       
 12  product_name       60247 non-null  string        
 13  category           60247 non-null  category      
 14  segmen

Kiểu dữ liệu của các biến hiện tại đã phù hợp. Tuy nhiên, cần kiểm tra các giá trị unique trong các biến này để đảm bảo không có giá trị trùng lặp ý nghĩa hoặc các giá trị không mang ý nghĩa.

In [54]:
for val in cols_to_category:
    print(f"Total unique values of '{val}' column: {df_inventory[val].nunique()}")
    counts = df_inventory[val].value_counts()
    percents = df_inventory[val].value_counts(normalize=True) * 100
    summary_df = pd.DataFrame({
        'Count': counts,
        'Percentage': percents.apply(lambda x: f"{x:.2f}%".replace('.', ','))
    })
    print(summary_df.to_string())
    print("-" * 40, "\n")

Total unique values of 'category' column: 4
            Count Percentage
category                    
Streetwear  31020     51,49%
Outdoor     21050     34,94%
GenZ         4674      7,76%
Casual       3503      5,81%
---------------------------------------- 

Total unique values of 'segment' column: 8
             Count Percentage
segment                      
Activewear   18290     30,36%
Everyday     13598     22,57%
Performance   7673     12,74%
Balanced      6622     10,99%
Trendy        4674      7,76%
Premium       3182      5,28%
Standard      3127      5,19%
All-weather   3081      5,11%
---------------------------------------- 



Hiện tại, các giá trị unique trong các biến đều có vẻ hợp lý và mang ý nghĩa. Không cần chỉnh sửa thêm

#### 2.1.13 Web traffic

In [55]:
cols_to_category = ['traffic_source']
cols_to_datetime = ['date']

df_web_traffic[cols_to_category] = df_web_traffic[cols_to_category].astype('category')

for col in cols_to_datetime:
    df_web_traffic[col] = pd.to_datetime(df_web_traffic[col], format='%Y-%m-%d', errors='coerce')

print("Display structure of data after data type casting: \n")
df_web_traffic.info()

Display structure of data after data type casting: 



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3652 entries, 0 to 3651
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   date                      3652 non-null   datetime64[ns]
 1   sessions                  3652 non-null   int64         
 2   unique_visitors           3652 non-null   int64         
 3   page_views                3652 non-null   int64         
 4   bounce_rate               3652 non-null   float64       
 5   avg_session_duration_sec  3652 non-null   float64       
 6   traffic_source            3652 non-null   category      
dtypes: category(1), datetime64[ns](1), float64(2), int64(3)
memory usage: 175.1 KB


Kiểu dữ liệu của các biến hiện tại đã phù hợp. Tuy nhiên, cần kiểm tra các giá trị unique trong các biến này để đảm bảo không có giá trị trùng lặp ý nghĩa hoặc các giá trị không mang ý nghĩa.

In [56]:
for val in cols_to_category:
    print(f"Total unique values of '{val}' column: {df_web_traffic[val].nunique()}")
    counts = df_web_traffic[val].value_counts()
    percents = df_web_traffic[val].value_counts(normalize=True) * 100
    summary_df = pd.DataFrame({
        'Count': counts,
        'Percentage': percents.apply(lambda x: f"{x:.2f}%".replace('.', ','))
    })
    print(summary_df.to_string())
    print("-" * 40, "\n")

Total unique values of 'traffic_source' column: 6
                Count Percentage
traffic_source                  
organic_search   1090     29,85%
paid_search       784     21,47%
social_media      632     17,31%
email_campaign    505     13,83%
referral          375     10,27%
direct            266      7,28%
---------------------------------------- 



Hiện tại, các giá trị unique trong các biến đều có vẻ hợp lý và mang ý nghĩa. Không cần chỉnh sửa thêm

### 2.2 Xử lý dữ liệu bị thiếu

#### 2.2.1 Promotional programs

In [57]:
df_promotions['applicable_category'] = df_promotions['applicable_category'].astype('string').fillna('All').astype('category')
df_promotions.info()

print("-" * 40, "\n")
print(f"Total unique values of 'applicable_category' column: {df_promotions['applicable_category'].nunique()}")
counts = df_promotions['applicable_category'].value_counts()
percents = df_promotions['applicable_category'].value_counts(normalize=True) * 100
summary_df = pd.DataFrame({
    'Count': counts,
    'Percentage': percents.apply(lambda x: f"{x:.2f}%".replace('.', ','))
})
print(summary_df.to_string())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   promo_id             50 non-null     string        
 1   promo_name           50 non-null     string        
 2   promo_type           50 non-null     category      
 3   discount_value       50 non-null     float64       
 4   start_date           50 non-null     datetime64[ns]
 5   end_date             50 non-null     datetime64[ns]
 6   applicable_category  50 non-null     category      
 7   promo_channel        50 non-null     category      
 8   stackable_flag       50 non-null     bool          
 9   min_order_value      50 non-null     int64         
dtypes: bool(1), category(3), datetime64[ns](2), float64(1), int64(1), string(2)
memory usage: 3.1 KB
---------------------------------------- 

Total unique values of 'applicable_category' column: 3
             

#### 2.2.2 Order detail items

In [58]:
col_to_check = ['promo_id', 'promo_id_2']
df_order_items[col_to_check] = df_order_items[col_to_check].fillna('None')
df_order_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 714669 entries, 0 to 714668
Data columns (total 7 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   order_id         714669 non-null  string 
 1   product_id       714669 non-null  string 
 2   quantity         714669 non-null  int64  
 3   unit_price       714669 non-null  float64
 4   discount_amount  714669 non-null  float64
 5   promo_id         714669 non-null  string 
 6   promo_id_2       714669 non-null  string 
dtypes: float64(2), int64(1), string(4)
memory usage: 38.2 MB


## Phase 3: Data Integration & Export

Trong tài liệu này, toàn bộ dữ liệu gốc từ BTC đã được đi qua các bước Data Profiling, chuẩn hóa kiểu dữ liệu, cũng như xử lý các giá trị trùng lặp và khuyết thiếu cơ bản. Trước khi lưu trữ thành bộ dữ liệu sạch (Cleaned Dataset), chúng ta sẽ thực hiện một bước tinh gọn cấu trúc.

### 3.1. Data Integration
Thông qua phân tích quan hệ thực thể, nhận thấy bảng *orders.csv* và bảng *payments.csv* có mối quan hệ 1-1 (mỗi đơn hàng có một mã thanh toán tương ứng). Để tối ưu hóa không gian lưu trữ và tránh việc phải dùng lệnh `merge` lặp lại nhiều lần ở các Notebook phân tích (EDA) phía sau, hai bảng này sẽ được tích hợp làm một.

In [59]:
df_payments = df_payments.drop(columns=['payment_method'])
df_orders = df_orders.merge(df_payments, on='order_id', how='left')
df_orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 10 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   order_id        646945 non-null  string        
 1   order_date      646945 non-null  datetime64[ns]
 2   customer_id     646945 non-null  string        
 3   zip             646945 non-null  string        
 4   order_status    646945 non-null  category      
 5   payment_method  646945 non-null  category      
 6   device_type     646945 non-null  category      
 7   order_source    646945 non-null  category      
 8   payment_value   646945 non-null  float64       
 9   installments    646945 non-null  category      
dtypes: category(5), datetime64[ns](1), float64(1), string(3)
memory usage: 27.8 MB


### 3.2. Xuất dữ liệu (Export Data)
Toàn bộ dữ liệu sau khi làm sạch sẽ được lưu vào thư mục *../dataset/cleaned/* với định dạng *.parquet*. 

**Lý do chọn định dạng Parquet:** giúp nén giảm dung lượng file và bảo toàn tuyệt đối các định dạng dữ liệu (như `datetime`, `category`, `string`) đã được chuẩn hóa ở Phase 2, giúp các thành viên trong team có thể trực tiếp vẽ biểu đồ ngay khi load data.

In [60]:
DATA_PATH = '../dataset/02_after_clean/'
os.makedirs('../dataset/02_after_clean/', exist_ok=True)

print("Exporting cleaned data...")

# Master
df_products.to_parquet(DATA_PATH + 'products.parquet', engine='pyarrow')
df_customers.to_parquet(DATA_PATH + 'customers.parquet')
df_promotions.to_parquet(DATA_PATH + 'promotions.parquet')
df_geography.to_parquet(DATA_PATH + 'geography.parquet')

# Transaction
df_orders.to_parquet(DATA_PATH + 'orders.parquet')
df_order_items.to_parquet(DATA_PATH + 'order_items.parquet')
df_shipments.to_parquet(DATA_PATH + 'shipments.parquet')
df_returns.to_parquet(DATA_PATH + 'returns.parquet')
df_reviews.to_parquet(DATA_PATH + 'reviews.parquet')

# Analytical
df_sales.to_parquet(DATA_PATH + 'sales.parquet')
#df_sample_submission.to_parquet(DATA_PATH + 'sample_submission.parquet')

# Operational
df_inventory.to_parquet(DATA_PATH + 'inventory.parquet')
df_web_traffic.to_parquet(DATA_PATH + 'web_traffic.parquet')

print("Export data successfully")

Exporting cleaned data...


Export data successfully


### 3.3. Instructions for importing data for subsequent notebooks.
Từ giai đoạn sau, khi cần gọi dữ liệu ra để trực quan hóa hoặc chạy mô hình, sử dụng hàm `read_parquet` của Pandas với đường dẫn trỏ về thư mục *cleaned*.

Ví dụ cụ thể:
```python
import pandas as pd
# Load dữ liệu khách hàng đã làm sạch
df_customerstomers = pd.read_parquet('../dataset/cleaned/customers.parquet').